# 🧠 推理服务 GPU 显存布局 — 从单模型到多模态与工具调用

**前置阅读**：建议先读完 `01-transformer-inference-primer.ipynb`（理解 KV Cache 基本概念）和 `02-quantization-basics.ipynb`（理解权重量化）。

**本文目标**：建立推理服务 GPU 显存的完整心智模型——从基础的显存层次到多模态、工具调用这些高级场景下的显存变化。读完这篇你会理解：

- GPU 的两种内存（HBM 和 SRAM）各自干什么、为什么推理是 memory-bound
- 一个推理请求的显存全景图：权重、KV Cache、激活值、临时 buffer 各占多少
- **多模态模型**：图像 token 如何让 KV Cache 爆炸
- **工具调用**：函数定义 + 多轮对话如何在显存中积累
- 推理框架的显存管理思路：预分配、对象池、prefix caching、offloading

## 1. GPU 显存层次：为什么推理是 memory-bound

### 1.1 两种内存，两个世界

```
┌─────────────────────────────────────────────────────────────┐
│                    NVIDIA GPU 内存层次                        │
├─────────────────────┬───────────────────────────────────────┤
│      HBM (显存)       │          SRAM (片上缓存)               │
│  High Bandwidth       │    Static RAM (L1 / Shared Memory)    │
│  Memory               │                                       │
├─────────────────────┼───────────────────────────────────────┤
│ 容量: 40-80 GB        │ 容量: ~40 MB (A100: 40MB L2)           │
│ 带宽: 1.5-3 TB/s      │ 带宽: ~19 TB/s (A100 SM 总量)          │
│ 延迟: ~数百 cycles     │ 延迟: ~数十 cycles                     │
│ 作用: 存权重/KV Cache  │ 作用: 存当前计算的临时数据              │
├─────────────────────┼───────────────────────────────────────┤
│ 类比:                 │ 类比:                                 │
│ 大仓库，容量大但        │ 工作台，就在手边但面积小                │
│ 每次取东西要走很远      │                                       │
└─────────────────────┴───────────────────────────────────────┘

实际数据（A100-80GB SXM）:
  HBM 带宽: 2,039 GB/s ≈ 2 TB/s
  SRAM 带宽: 19,500 GB/s ≈ 19 TB/s (所有 SM 聚合)
  → SRAM 比 HBM 快 ~10x，但容量只有 ~0.05%
```

### 1.2 推理时的数据流

每次 Decode 一个 token 的数据流动：

```
1. 从 HBM 读权重: ~14 GB (LLaMA-7B FP16)
                ↓
2. 暂存 SRAM  →  做矩阵乘法 (Tensor Core)
                ↓
3. 从 HBM 读 KV Cache: ~seq_len × 512KB  (当前序列的历史 K,V)
                ↓
4. 暂存 SRAM  →  做 Attention
                ↓
5. 新 K,V 写回 HBM (追加到 KV Cache)
                ↓
6. 输出 logits → HBM → 传回 CPU → 采样 → 得到下一个 token

关键观察:
  - 每一步都要从 HBM 读大量数据
  - Tensor Core 算得飞快，但数据从 HBM 来"太慢"
  - decode 阶段 1 个 token 的计算量很小, 但要从 HBM 读整个 KV Cache
  → 这就是 Decode 是 memory-bound 的根源
```

### 1.3 显存带宽的实操感知

```python
# A100-80GB: HBM 带宽 2039 GB/s
# 一个 Decode step 需要读的数据量:
#   模型权重: 14 GB (LLaMA-7B FP16)
#   KV Cache: seq_len × 0.5 MB  (假设 seq_len=4096 → 2 GB)
#   激活值等: ~0.5 GB
#   总计: ~16.5 GB / step

# 如果只受带宽限制:
#   理论最快 = 2039 / 16.5 ≈ 123 steps/s ≈ 123 tokens/s

# 实际远低于这个值, 因为:
#   1. 不是所有时间都在做连续大块读取 (随机访问、小数据块)
#   2. Attention 的 softmax 等操作在 SRAM 和 HBM 间反复搬运
#   3. 多个请求会争抢带宽 (concurrent requests)
```

## 2. 推理请求的显存全景图

### 2.1 一个完整的显存地图

```
┌──────────────────────────────────────────────────────────────────┐
│                      推理服务显存布局 (A100-80GB)                   │
├─────────────┬─────────────┬──────────────┬───────────────┬───────┤
│  模型权重     │  KV Cache   │   激活值      │  临时 Buffer   │ 系统  │
│  (持久占用)   │ (动态增长)   │ (当前批次)     │  (CUDA/框架)   │ 开销  │
├─────────────┼─────────────┼──────────────┼───────────────┼───────┤
│ ~14 GB       │ 0-60 GB     │ ~1-4 GB       │ ~2-4 GB        │ ~2 GB │
│ (LLaMA-7B)   │ (随请求变)   │ (batch 相关)  │ (上下文/图等)   │       │
├─────────────┴─────────────┴──────────────┴───────────────┴───────┤
│                                                                   │
│  固定部分 (模型加载后不变):                                        │
│    ┌──────────────────────┐                                       │
│    │ 模型权重 (含量化)      │  ← 加载时分配, 生命周期 = 整个服务进程   │
│    │ 如果 Q4_K_M: ~4 GB   │                                       │
│    │ 如果 FP16:    ~14 GB │                                       │
│    │ CUDA graph 缓存      │  ← TensorRT-LLM 额外占 ~1-3 GB         │
│    │ NCCL 通信 buffer     │  ← 分布式推理时额外占                   │
│    └──────────────────────┘                                       │
│                                                                   │
│  动态部分 (随请求变化):                                            │
│    ┌──────────────────────────────────────────┐                   │
│    │ KV Cache                                  │                   │
│    │  • 每个请求: seq_len × 2 × n_layers ×       │                   │
│    │              n_kv_heads × head_dim × dtype │                   │
│    │  • 10 并发 4K: ~20 GB                      │                   │
│    │  • 50 并发 4K: ~100 GB → OOM!              │                   │
│    │  • 支持 prefix caching 时部分可共享        │                   │
│    ├──────────────────────────────────────────┤                   │
│    │ 激活值 (Activations)                       │                   │
│    │  • 当前 batch 的中间计算结果                │                   │
│    │  • Batch size 越大, 激活值越多              │                   │
│    │  • FlashAttention 会额外用 SRAM 做暂存     │                   │
│    ├──────────────────────────────────────────┤                   │
│    │ 临时 Buffer (PyTorch/CUDA allocator)       │                   │
│    │  • PyTorch caching allocator 的碎片        │                   │
│    │  • NCCL 通信临时 buffer                    │                   │
│    │  • GPU kernel launch 的参数/临时数组       │                   │
│    └──────────────────────────────────────────┘                   │
└──────────────────────────────────────────────────────────────────┘
```

### 2.2 显存变化的时间线

```
请求到达时间线:

t=0    服务启动
       [权重 ████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 14/80 GB

t=1    第一个请求到达 (prompt=500 tokens)
       [权重 ████████████████][KV █░░░][临时 ░░] ~16/80 GB
                              ≈0.25 GB per 500 tokens

t=2    请求生成中, seq_len 增长到 2000
       [权重 ████████████████][KV ████░][临时 ░░] ~17/80 GB
                              ≈1 GB per 2000 tokens

t=3    10 个并发请求, 且请求 5 收到一个超长 context (10K tokens)
       [权重 ████████████████][KV ████████████████████████░░░]
       14 GB + (9×2K + 1×10K) × 0.5 MB = 14 + 14 = 28 GB

t=4    请求 5 生成到 12K tokens
       [权重 ████████████████][KV ████████████████████████████]
       14 + 16 = 30 GB → 还在安全线内...

t=5    新增 3 个请求, seq_len=4K each
       [权重 ████████████████][KV ████████████████████████████████████]
       14 + 22 = 36 GB → 危险, 触发 OOM 风险

       框架应对:
       • vLLM: 拒绝新请求 (preemption → swap KV Cache 到 CPU)
       • TensorRT-LLM: 挂起低优请求
       • llama.cpp: 返回错误信息
```

## 3. 实验：显存计算的交互式模型

下面的代码让你可以调参观察不同场景下的显存分布：

In [1]:
# 推理服务显存计算器

import math

class GPU_Memory_Model:
    def __init__(self, total_gb=80):
        self.total_gb = total_gb
    
    def weights_gb(self, params_billions, dtype="fp16", quant=None):
        """模型权重显存"""
        bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2}
        base = params_billions * 1e9 * bytes_per_param[dtype] / (1024**3)
        
        quant_factors = {
            "fp16": 1.0, "fp8": 0.5, "int8": 0.5,
            "int4": 0.25,  "q4_k_m": 0.3, "q5_k_m": 0.35,
            "q8_0": 0.5, "awq_int4": 0.25, "gptq_int4": 0.25
        }
        factor = quant_factors.get(quant, 1.0) if quant else 1.0
        return base * factor
    
    def kv_cache_per_token_gb(self, n_layers, d_model, n_kv_heads, 
                                kv_dtype="fp16"):
        """每个 token 的 KV Cache 大小"""
        head_dim = d_model // (d_model // 128)  # simplified
        bytes_per = {"fp16": 2, "fp8": 1, "int8": 1, "q8_0": 1, "q4_0": 0.5}
        bpe = bytes_per.get(kv_dtype, 2)
        per_token = 2 * n_layers * n_kv_heads * head_dim * bpe
        return per_token / (1024**3)
    
    def kv_cache_total_gb(self, n_layers, d_model, n_kv_heads,
                           seq_len, n_concurrent, kv_dtype="fp16"):
        """总 KV Cache"""
        return self.kv_cache_per_token_gb(n_layers, d_model, n_kv_heads, kv_dtype)                * seq_len * n_concurrent
    
    def activations_gb(self, batch_size, d_model, n_layers, seq_len_avg):
        """激活值估算 (粗略)"""
        # 粗略公式: batch × seq × d_model × n_layers × 2 bytes × overhead
        return batch_size * seq_len_avg * d_model * n_layers * 2 * 1.2 / (1024**3)
    
    def system_overhead_gb(self):
        """CUDA context, 框架开销, 碎片等"""
        return 2.0  # 大约 2GB
    
    def report(self, name, params_b, n_layers, d_model, n_kv_heads,
               seq_len, n_concurrent, quant=None, kv_dtype="fp16",
               extra_encoder_gb=0, tool_overhead_gb=0):
        
        w = self.weights_gb(params_b, quant=quant)
        kv = self.kv_cache_total_gb(n_layers, d_model, n_kv_heads,
                                     seq_len, n_concurrent, kv_dtype)
        act = self.activations_gb(n_concurrent, d_model, n_layers, seq_len)
        sys_oh = self.system_overhead_gb()
        
        total = w + kv + act + sys_oh + extra_encoder_gb + tool_overhead_gb
        free = self.total_gb - total
        
        bar_len = 50
        used_pct = total / self.total_gb
        
        print(f"\n{'='*70}")
        print(f"  {name}")
        print(f"{'='*70}")
        print(f"  模型权重:      {w:>8.1f} GB")
        print(f"  KV Cache:      {kv:>8.1f} GB  ({n_concurrent}并发 × {seq_len} tokens)")
        if extra_encoder_gb > 0:
            print(f"  Visual Encoder:{extra_encoder_gb:>8.1f} GB  (多模态)")
        if tool_overhead_gb > 0:
            print(f"  工具调用开销:  {tool_overhead_gb:>8.1f} GB  (函数定义+多轮)")
        print(f"  激活值:        {act:>8.1f} GB")
        print(f"  系统开销:      {sys_oh:>8.1f} GB")
        print(f"  {'─'*40}")
        print(f"  总计:          {total:>8.1f} GB / {self.total_gb} GB")
        print(f"  剩余:          {free:>8.1f} GB  {'⚠️  OOM 风险!' if free < 5 else '✅ 安全' if free > 20 else '⚠️  紧张'}")
        
        # 内存条
        filled = int(bar_len * used_pct)
        bar = '█' * filled + '░' * (bar_len - filled)
        print(f"  [{bar}] {used_pct*100:.0f}%")
        
        return total, free


m = GPU_Memory_Model(total_gb=80)  # A100-80GB

# ── 场景 1: 基础文本推理 ──
m.report("场景 1: LLaMA-7B 文本推理 (FP16)",
         params_b=7, n_layers=32, d_model=4096, n_kv_heads=32,
         seq_len=4096, n_concurrent=10, quant="fp16")

m.report("场景 2: LLaMA-7B 文本推理 (AWQ INT4)",
         params_b=7, n_layers=32, d_model=4096, n_kv_heads=32,
         seq_len=4096, n_concurrent=10, quant="awq_int4")

m.report("场景 3: LLaMA-70B 文本推理 (FP16)",
         params_b=70, n_layers=80, d_model=8192, n_kv_heads=64,
         seq_len=4096, n_concurrent=5, quant="fp16")

m.report("场景 4: LLaMA-70B + GQA + AWQ INT4",
         params_b=70, n_layers=80, d_model=8192, n_kv_heads=8,  # GQA: 8 KV heads
         seq_len=4096, n_concurrent=10, quant="awq_int4")

# ── KV Cache 量化效果 ──
print(f"\n{'='*70}")
print(f"  KV Cache 量化的收益 (LLaMA-7B, 10并发, 4K tokens)")
print(f"{'='*70}")
for kv_type in ["fp16", "fp8", "q8_0", "q4_0"]:
    kv = m.kv_cache_total_gb(32, 4096, 32, 4096, 10, kv_type)
    delta = m.kv_cache_total_gb(32, 4096, 32, 4096, 10, "fp16") - kv
    print(f"  {kv_type:>6s}: {kv:.1f} GB  (节省 {delta:.1f} GB)")

print(f"\n  💡 关键发现: KV Cache Q8_0 几乎无损, 省一半显存!")


  场景 1: LLaMA-7B 文本推理 (FP16)
  模型权重:          13.0 GB
  KV Cache:          20.0 GB  (10并发 × 4096 tokens)
  激活值:            12.0 GB
  系统开销:           2.0 GB
  ────────────────────────────────────────
  总计:              47.0 GB / 80 GB
  剩余:              33.0 GB  ✅ 安全
  [█████████████████████████████░░░░░░░░░░░░░░░░░░░░░] 59%

  场景 2: LLaMA-7B 文本推理 (AWQ INT4)
  模型权重:           3.3 GB
  KV Cache:          20.0 GB  (10并发 × 4096 tokens)
  激活值:            12.0 GB
  系统开销:           2.0 GB
  ────────────────────────────────────────
  总计:              37.3 GB / 80 GB
  剩余:              42.7 GB  ✅ 安全
  [███████████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░] 47%

  场景 3: LLaMA-70B 文本推理 (FP16)
  模型权重:         130.4 GB
  KV Cache:          50.0 GB  (5并发 × 4096 tokens)
  激活值:            30.0 GB
  系统开销:           2.0 GB
  ────────────────────────────────────────
  总计:             212.4 GB / 80 GB
  剩余:            -132.4 GB  ⚠️  OOM 风险!
  [████████████████████████████████████████████████████████████████

## 4. 多模态模型：当图像进入推理

### 4.1 图像如何变成"token"

多模态模型 (LLaVA, Qwen-VL, InternVL, GPT-4V 架构) 的处理流程：

```
输入: 一张 1024×1024 的图片 + "请描述这张图"
       │
       ▼
┌─────────────────────────────┐
│  Visual Encoder (ViT/CLIP)   │  ← 独立模型, 额外显存!
│  图 → patch → embedding      │
│  1024² → (1024/14)² = 5184   │
│  patches × 1024-dim =        │
│  [5184, 1024] visual tokens   │
│                               │
│  经过 projection:             │
│  [5184, 1024] → [576, 4096]  │  ← 压缩到 LLM 维度
│  (通常会做 spatial pooling)   │
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  拼接: [visual_tokens | text_tokens]
│  [576 img tokens] + [5 text tokens] = 581 tokens
│              │
│              ▼
│  LLM Decoder (标准 Transformer)
│  为这 581 个 token 生成 KV Cache
└─────────────────────────────┘
```

### 4.2 多模态的显存冲击 — 这才是重点

```
关键问题: 图像 token 数量巨大!

  文本 prompt:  "请描述这张图"   → 5 tokens     → KV Cache ~2.5 KB
  一张图片:      576 visual tokens             → KV Cache ~288 KB
  10 张图片:     5,760 visual tokens           → KV Cache ~2.8 MB (per request!)
  
  10 个并发请求, 每请求 5 张图:
    Visual tokens: 10 × 5 × 576 = 28,800 tokens
    KV Cache: 28,800 × 0.5 MB/1K tokens ≈ 14.4 GB
    仅 visual tokens 的 KV Cache 就 14.4 GB!

对比:
  纯文本 10 并发 4K:  KV Cache ≈ 20 GB
  5 图 10 并发:       KV Cache ≈ 14.4 (visual) + 2 (text) ≈ 16.4 GB
  10 图 10 并发:      KV Cache ≈ 28.8 + 2 ≈ 30.8 GB  → 危险!
```

### 4.3 Visual Encoder 的独立显存

```
┌────────────────────────────────────────────────────┐
│              多模态模型的显存全景                      │
├────────────────────────────────────────────────────┤
│                                                     │
│  ┌──────────────────┐  ┌─────────────────────────┐ │
│  │ Visual Encoder    │  │ LLM Decoder              │ │
│  │ (ViT-L/CLIP)      │  │ (LLaMA-7B)               │ │
│  │                    │  │                          │ │
│  │ 权重: ~1.8 GB      │  │ 权重: ~14 GB (FP16)       │ │
│  │ (FP16, 计算后      │  │                          │ │
│  │  可 offload)       │  │ KV Cache: 0-60 GB        │ │
│  │                    │  │                          │ │
│  │ Visual KV Cache:   │  │ 跨模态 Projector: 0.1 GB │ │
│  │  • 大图 → 多 token │  │                          │ │
│  │  • 只在 prefill     │  │                          │ │
│  │    阶段存在         │  │                          │ │
│  └──────────────────┘  └─────────────────────────┘ │
│                                                     │
│  典型分配 (LLaVA-7B, A100-80GB):                     │
│    权重: 14 (LLM) + 1.8 (ViT) + 0.1 (Proj) = 15.9 GB│
│    KV Cache: 视请求而定                               │
│    激活值: ~2-4 GB (含 visual feature map)            │
│    剩余给 KV Cache 的空间: 80 - 16 - 3 - 2 ≈ 59 GB   │
└────────────────────────────────────────────────────┘
```

### 4.4 视频/音频的多模态

```
视频 = 多帧图像:
  1 分钟视频 @ 1 fps = 60 帧
  60 × 576 visual tokens = 34,560 tokens
  → 仅 prefill 就需要处理 34K tokens！
  → KV Cache per request ≈ 17 MB

音频 (Whisper-style):
  30 秒音频 → 经过 encoder → ~1500 audio tokens
  → KV Cache ≈ 0.75 MB per request

多模态组合 (GPT-4V style):
  10 张图 + 30 秒音频 + 文本指令:
  Visual tokens: 10 × 576 = 5,760
  Audio tokens:  1,500
  Text tokens:    500
  Total:          ~7,760 tokens
  → KV Cache per request ≈ 3.9 MB
  → 10 并发: ~39 GB 仅 KV Cache!
```

## 5. 工具调用：函数定义与多轮对话的显存暗流

### 5.1 工具调用为什么是显存杀手

工具调用 (Function Calling / Tool Use) 会从三个维度冲击显存：

```
┌──────────────────────────────────────────────────┐
│           工具调用的显存冲击 (三个维度)              │
├────────────┬─────────────────────────────────────┤
│ 1. 函数定义  │ system prompt 里塞所有 tool schema    │
│             │  → prefill 阶段就要处理这些 token       │
├────────────┼─────────────────────────────────────┤
│ 2. 多轮交互  │ 每轮: user→assistant(tool_call)→     │
│             │       tool_result→assistant→...       │
│             │  → KV Cache 在多轮间持续膨胀!           │
├────────────┼─────────────────────────────────────┤
│ 3. 工具结果  │ tool result 可能非常大!                │
│             │  → 代码搜索结果、数据库查询结果等        │
│             │  → 突然注入大量 token 到 context       │
└────────────┴─────────────────────────────────────┘
```

### 5.2 函数定义的显存代价

```
典型的 function definitions 大小:

  简单 (1-2 个工具):
    "你有一个函数 get_weather(city: str) → {...}"
    ≈ 80-150 tokens

  中等 (5-10 个工具):
    OpenAI function calling format, 每个 tool 有 name/description/parameters
    ≈ 500-1500 tokens

  Agent 框架 (20+ 工具):
    LangChain/LlamaIndex Agent, 大量 tool schema
    ≈ 2000-5000 tokens

  MCP Server (复杂工具集):
    文件系统 + 数据库 + API + ...
    ≈ 5000-15000 tokens in system prompt!

  显存影响:
    函数定义 5000 tokens → prefill KV Cache ≈ 2.5 MB per request
    10 并发 → 25 MB — 还好...

    BUT: 这些 token 每个请求都一样 → 应该用 prefix caching 共享!
```

### 5.3 多轮工具调用的 KV Cache 膨胀

```
一个完整的工具调用会话:

  Round 1:
    User: "北京今天天气怎么样?"                        → 6 tokens
    Assistant: tool_call(get_weather, {city: "北京"})  → 12 tokens
    Tool Result: '{"city":"北京","temp":35,"humidity":80,...}' → 80 tokens
    Assistant: "北京今天 35°C, 湿度 80%..."              → 30 tokens
    ─────────────────────────────────────────────────
    累计: 128 tokens → KV Cache: ~64 KB

  Round 2:
    User: "那明天呢?"                                  → 4 tokens
    Assistant: tool_call(get_weather, {city: "北京", date: "2026-07-30"}) → 15 tokens
    Tool Result: '{"city":"北京","temp":32,...}'        → 70 tokens
    Assistant: "明天 32°C..."                           → 20 tokens
    ─────────────────────────────────────────────────
    累计: 237 tokens → KV Cache: ~118 KB

  Round 10 (长对话, 多次工具调用):
    累计: 2500+ tokens → KV Cache: ~1.2 MB per request

  10 个这样的并发请求:
    KV Cache: 10 × 1.2 MB ≈ 12 MB  ← 好像不多?

  BUT: 实际场景中, 工具调用往往伴随长 context:
    • 代码生成 Agent: 读取整个文件(5000+ tokens)作为 tool result
    • RAG 搜索: 返回 10 个文档片段(3000+ tokens)
    • 数据分析: SQL 结果 200 行(4000+ tokens)
    
  真实场景 (10 并发, 每请求 5 轮工具调用):
    函数定义: 2000 tokens
    对话历史: 5 轮 × (user ~20 + assistant ~50 + tool ~2000) = 10,350 tokens
    总计: ~12,350 tokens per request
    KV Cache: 12,350 × 0.5 KB ≈ 6.2 MB per request
    10 并发: 62 GB 仅 KV Cache!  ← 这才是现实
```

## 6. 工具调用的显存实验

In [2]:
# 工具调用场景的显存模拟

class ToolCallMemorySim:
    """模拟工具调用对 KV Cache 的影响"""
    
    def __init__(self, model_gb=14, total_gb=80, per_token_kv_kb=0.5):
        self.model_gb = model_gb
        self.total_gb = total_gb
        self.per_token_kv_kb = per_token_kv_kb  # per token KV Cache
    
    def simulate_session(self, func_def_tokens, rounds, 
                         user_tokens_per_round=20,
                         assistant_tokens_per_round=50,
                         tool_result_tokens_per_round=2000):
        """模拟一个工具调用会话的 token 累积"""
        total_tokens = func_def_tokens  # 函数定义在 system prompt
        
        print(f"  函数定义: {func_def_tokens} tokens")
        print(f"  {'─'*40}")
        
        for r in range(1, rounds + 1):
            total_tokens += user_tokens_per_round
            total_tokens += assistant_tokens_per_round  # tool_call
            total_tokens += tool_result_tokens_per_round
            total_tokens += assistant_tokens_per_round  # final response
            
            kv_mb = total_tokens * self.per_token_kv_kb / 1024
            print(f"  Round {r}: 累计 {total_tokens:>6,} tokens, "
                  f"KV Cache: {kv_mb:.1f} MB")
        
        return total_tokens
    
    def report_scenario(self, name, func_def_tokens, rounds, n_concurrent,
                        tool_result_tokens=2000):
        print(f"\n{'─'*60}")
        print(f"  {name}")
        print(f"{'─'*60}")
        tokens = self.simulate_session(func_def_tokens, rounds,
                                        tool_result_tokens_per_round=tool_result_tokens)
        
        kv_per_req_gb = tokens * self.per_token_kv_kb / (1024**2)
        kv_total_gb = kv_per_req_gb * n_concurrent
        total_gb = self.model_gb + kv_total_gb + 3  # +3GB 激活值+系统
        
        print(f"  {'─'*40}")
        print(f"  每请求 token 数: {tokens:,}")
        print(f"  每请求 KV Cache: {kv_per_req_gb:.1f} GB")
        print(f"  {n_concurrent} 并发 KV Cache: {kv_total_gb:.1f} GB")
        print(f"  总显存: {total_gb:.1f} GB / {self.total_gb} GB")
        
        if total_gb > self.total_gb:
            print(f"  ❌ OOM! 超出 {(total_gb - self.total_gb):.1f} GB")
        elif total_gb > self.total_gb * 0.85:
            print(f"  ⚠️  危险! 仅剩 {self.total_gb - total_gb:.1f} GB")
        else:
            print(f"  ✅ 安全, 剩余 {self.total_gb - total_gb:.1f} GB")
        
        return tokens, kv_total_gb


sim = ToolCallMemorySim(model_gb=14, total_gb=80)

# 场景 1: 简单工具调用
sim.report_scenario("场景 A: 简单 Agent (3工具, 5轮, 10并发)",
    func_def_tokens=800, rounds=5, n_concurrent=10,
    tool_result_tokens=500)

# 场景 2: 代码 Agent
sim.report_scenario("场景 B: 代码 Agent (10工具, 10轮, 10并发, 读取源码)",
    func_def_tokens=3000, rounds=10, n_concurrent=10,
    tool_result_tokens=5000)  # 读取文件内容

# 场景 3: RAG Agent
sim.report_scenario("场景 C: RAG Agent (5工具, 3轮, 20并发, 文档检索)",
    func_def_tokens=1500, rounds=3, n_concurrent=20,
    tool_result_tokens=3000)  # 返回文档片段

# 场景 4: MCP 复杂 Agent
sim.report_scenario("场景 D: MCP Agent (30工具, 8轮, 5并发)",
    func_def_tokens=8000, rounds=8, n_concurrent=5,
    tool_result_tokens=2000)

print(f"\n{'='*60}")
print(f"  关键洞察:")
print(f"  • 函数定义占用不算多, 但用 prefix caching 白赚的优化")
print(f"  • 真正杀手是 tool_result — 代码/文档/数据库结果")
print(f"  • 多轮对话让 KV Cache 线性累积, 没有回收机制")
print(f"  • Agent 框架需要特别关注 context 管理和 KV Cache 预算")


────────────────────────────────────────────────────────────
  场景 A: 简单 Agent (3工具, 5轮, 10并发)
────────────────────────────────────────────────────────────
  函数定义: 800 tokens
  ────────────────────────────────────────
  Round 1: 累计  1,420 tokens, KV Cache: 0.7 MB
  Round 2: 累计  2,040 tokens, KV Cache: 1.0 MB
  Round 3: 累计  2,660 tokens, KV Cache: 1.3 MB
  Round 4: 累计  3,280 tokens, KV Cache: 1.6 MB
  Round 5: 累计  3,900 tokens, KV Cache: 1.9 MB
  ────────────────────────────────────────
  每请求 token 数: 3,900
  每请求 KV Cache: 0.0 GB
  10 并发 KV Cache: 0.0 GB
  总显存: 17.0 GB / 80 GB
  ✅ 安全, 剩余 63.0 GB

────────────────────────────────────────────────────────────
  场景 B: 代码 Agent (10工具, 10轮, 10并发, 读取源码)
────────────────────────────────────────────────────────────
  函数定义: 3000 tokens
  ────────────────────────────────────────
  Round 1: 累计  8,120 tokens, KV Cache: 4.0 MB
  Round 2: 累计 13,240 tokens, KV Cache: 6.5 MB
  Round 3: 累计 18,360 tokens, KV Cache: 9.0 MB
  Round 4: 累计 23,480 tokens, KV C

## 7. 推理框架的显存管理策略

### 7.1 四大策略总览

```
┌─────────────────────────────────────────────────────────────────┐
│                   推理框架的显存管理策略                            │
├──────────────┬──────────────────────┬───────────────────────────┤
│    策略       │        原理           │        代表框架             │
├──────────────┼──────────────────────┼───────────────────────────┤
│              │                      │                           │
│ 分块管理      │ KV Cache 分成固定大小   │ vLLM (PagedAttention)     │
│ (Block-based) │ 的 block, 按需分配回收  │ TensorRT-LLM             │
│              │ 类比: OS 虚拟内存       │ SGLang (RadixAttention)  │
│              │                      │                           │
├──────────────┼──────────────────────┼───────────────────────────┤
│              │                      │                           │
│ 预分配 +      │ 启动时预估最大 KV Cache  │ llama.cpp                │
│ 静态管理      │ 分配一大块连续内存      │ Ollama                   │
│ (Pre-allocate)│ 优点: 简单, 无碎片管理  │                          │
│              │ 缺点: 内部碎片, 不灵活   │                          │
│              │                      │                           │
├──────────────┼──────────────────────┼───────────────────────────┤
│              │                      │                           │
│ 前缀缓存      │ 公共前缀的 KV Cache     │ SGLang (RadixAttention)  │
│ (Prefix      │ 多请求共享             │ vLLM (APC)               │
│  Caching)    │ system prompt / 工具定义│ TensorRT-LLM             │
│              │ 只存一份!              │                          │
│              │                      │                           │
├──────────────┼──────────────────────┼───────────────────────────┤
│              │                      │                           │
│ Swap /       │ KV Cache 换出到 CPU   │ vLLM (preemption)        │
│ Offloading   │ 内存甚至 SSD          │ llama.cpp (mmap)         │
│              │ 优点: 突破 GPU 显存限制 │ Ollama (自动换出)        │
│              │ 缺点: PCIe 带宽 ~50GB/s│                          │
│              │       远慢于 HBM      │                          │
│              │                      │                           │
└──────────────┴──────────────────────┴───────────────────────────┘
```

### 7.2 Prefix Caching 在多模态和工具调用中的价值

这是对多模态和工具调用**最有价值的优化**：

```
无 prefix caching:
  10 个请求, 每个有 2000 tokens 的函数定义 system prompt
  → 10 × 2000 = 20,000 tokens 的 KV Cache
  → 20,000 × 0.5 KB = 10 MB

有 prefix caching:
  2000 tokens 的函数定义只存一份 KV Cache
  → 2000 × 0.5 KB = 1 MB
  → 节省 90%！

多模态同理:
  10 个请求分析同一张图片:
  无 caching: 10 × 576 visual tokens = 5,760 tokens ≈ 2.9 MB KV Cache
  有 caching: 576 visual tokens ≈ 0.3 MB KV Cache
  → 节省 90%！

但现实中:
  • 函数定义完全相同 → 完美命中 cache ✓
  • 同一张图 → 完美命中 cache ✓
  • 相似但不完全相同的 prompt → cache miss ✗
  • 工具返回不同结果 → 后面部分 cache miss ✗
```

### 7.3 不同场景的最佳实践

```
场景: 纯文本 API 服务
├─ 框架: vLLM (PagedAttention)
├─ KV Cache: FP16 or FP8
├─ 关键优化: Continuous batching + prefix caching
└─ 显存配比: 权重 40% | KV Cache 50% | 其他 10%

场景: 本地消费级 GPU
├─ 框架: llama.cpp / Ollama
├─ 量化: Q4_K_M (权重) + Q8_0 (KV Cache)
├─ 关键优化: 量化是一切的基础
└─ 显存配比: 权重 25% | KV Cache 65% | 其他 10%

场景: 多模态服务
├─ 框架: vLLM (支持多模态) / SGLang
├─ 关键优化: Visual token compression, prefix caching for images
├─ 显存配比: 权重 35% (含 encoder) | KV Cache 55% | 其他 10%
└─ 特别注意: 限制单请求最大图片数, 设置 visual token 上限

场景: Agent / 工具调用
├─ 框架: vLLM + prefix caching
├─ 关键优化: 
│   • 函数定义放 system prompt → 完美命中 prefix cache
│   • 设置 max_tokens 上限防止无限循环
│   • 限制 tool_result 大小 (截断)
│   • 设置 max_rounds 防止 KV Cache 无限膨胀
├─ 显存配比: 权重 35% | KV Cache 55% | 其他 10%
└─ 特别注意: 用 context window 管理策略 (滑动窗口 / 摘要)


## 8. 总结：显存决策速查

```
你的场景是？

"我就跑个 7B 模型, 10 个并发"
├─ 最简方案: vLLM + 模型 AWQ INT4
├─ 显存: 权重 ~4GB + KV Cache ~15GB ≈ 19GB
└─ 甚至能在 RTX 4090 (24GB) 上跑

"我要跑多模态, 用户会上传图片"
├─ Visual encoder 额外占 ~2GB
├─ 限制每请求最多 5 张图
├─ 开启 visual token compression
└─ 显存规划: 权重 ~16GB + KV Cache ~40GB ≈ 足够 (A100)

"我要跑 Agent, 多轮工具调用"
├─ 函数定义必须用 prefix caching
├─ 限制 tool_result 大小 (最多 4000 tokens)
├─ 限制最大轮数 (最多 10 轮)
├─ 考虑滑动窗口: 超过 10 轮后丢弃最老的若干轮
└─ 显存规划: 权重 ~14GB + KV Cache ~50GB ≈ 足够 (A100)

"我只有 8GB 显存"
├─ llama.cpp + Q4_K_M
├─ KV Cache Q8_0
├─ 小 context (2048 tokens)
└─ 甚至能跑 7B 模型 + 1-2 并发
```

## 下一步

带着显存布局的完整理解进入各框架 deep dive：

- **llama.cpp** → 看它如何在有限显存下做静态分配和 KV Cache 量化
- **vLLM** → 看 PagedAttention 如何像 OS 虚拟内存一样管理 KV Cache block
- **SGLang** → 看 RadixAttention 如何用 radix tree 实现 prefix caching
- **TensorRT-LLM** → 看编译优化如何减少临时 buffer 和激活值开销